# Practical 12: Metropolis-Hastings Algorithm & Deterministic Solar Cycle Modeling

- **Part A:** Implements a log-posterior Metropolis sampler for a Binomial likelihood with a flat prior ($k = 21$ heads in $n = 30$ flips), validates sampled moments against the exact analytical $\text{Beta}(22, 10)$ posterior, and analyzes step-size sensitivity.
- **Part B:** Builds a deterministic harmonic wave model for monthly sunspot counts (`../Datasets/sunspots.csv`), evaluates residual diagnostics, and verifies deterministic forecast reproducibility.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

plt.style.use('seaborn-v0_8-whitegrid')

## Part A: Log-Posterior Metropolis Sampler (Beta-Binomial Model)

Given $k = 21$ successes in $n = 30$ Bernoulli trials with a uniform prior $p \sim \text{Uniform}(0, 1)$, the exact posterior is $p \mid k, n \sim \text{Beta}(22, 10)$. Evaluating the target in log-space $\ell(p) = 21 \log(p) + 9 \log(1 - p)$ avoids floating-point underflow.

In [ ]:
k_heads, n_flips = 21, 30

def log_target_beta(p):
    if p <= 0.0 or p >= 1.0:
        return -np.inf
    return k_heads * np.log(p) + (n_flips - k_heads) * np.log(1.0 - p)

def run_metropolis_1d(p_init=0.12, n_steps=12000, sigma=0.12, seed=42):
    rng = np.random.default_rng(seed)
    chain = np.empty(n_steps)
    cur_p = p_init
    cur_lp = log_target_beta(cur_p)
    accepted = 0

    for i in range(n_steps):
        cand_p = cur_p + rng.normal(0.0, sigma)
        cand_lp = log_target_beta(cand_p)
        if np.log(rng.uniform()) < (cand_lp - cur_lp):
            cur_p, cur_lp = cand_p, cand_lp
            accepted += 1
        chain[i] = cur_p

    return chain, accepted / n_steps

samples, acc_rate = run_metropolis_1d(p_init=0.12, n_steps=12000, sigma=0.12)
print(f'Acceptance Rate : {acc_rate:.3f}')
print(f'First 5 Draws   : {np.round(samples[:5], 4)}')

In [ ]:
burn_in = 800
kept_samples = samples[burn_in:]
exact_mean = (k_heads + 1) / (n_flips + 2)
exact_std = stats.beta.std(k_heads + 1, n_flips - k_heads + 1)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4))
axes[0].plot(samples, color='#2b5c8f', linewidth=0.55)
axes[0].axvline(burn_in, color='#d95f02', linestyle='--', label='Burn-in End')
axes[0].set_title('Trace Plot of Probability p', fontweight='bold')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('p')
axes[0].legend()

grid = np.linspace(0.01, 0.99, 350)
axes[1].hist(kept_samples, bins=40, density=True, color='#8da0cb', edgecolor='white', label='Metropolis Draws')
axes[1].plot(grid, stats.beta.pdf(grid, k_heads + 1, n_flips - k_heads + 1), color='#e7298a', linewidth=2.2, label='Exact Beta(22, 10)')
axes[1].set_title('Sampled Posterior vs. Exact Beta(22, 10)', fontweight='bold')
axes[1].set_xlabel('p')
axes[1].set_ylabel('Density')
axes[1].legend()
plt.tight_layout()
plt.show()

print(f'Sampled Mean : {kept_samples.mean():.4f} | Exact Mean : {exact_mean:.4f}')
print(f'Sampled Std  : {kept_samples.std():.4f} | Exact Std  : {exact_std:.4f}')

### Proposal Step-Width Sensitivity

In [ ]:
tuning_rows = []
for w in [0.01, 0.12, 0.45, 2.50]:
    s_w, r_w = run_metropolis_1d(p_init=0.12, n_steps=12000, sigma=w)
    tuning_rows.append({
        'Proposal Width': w,
        'Acceptance Rate': round(r_w, 3),
        'Sampled Mean': round(s_w[burn_in:].mean(), 4),
        'Abs Error vs Exact': round(abs(s_w[burn_in:].mean() - exact_mean), 4),
    })

print(pd.DataFrame(tuning_rows).to_string(index=False))
print(f'Exact Posterior Mean: {exact_mean:.4f}')

## Part B: Deterministic Harmonic Solar Cycle Model

A deterministic model contains zero stochastic randomness: identical inputs always produce identical outputs. We fit a four-parameter squared-sinusoid wave:

$$\text{count}(t) = c_0 + A \sin^2\!\left(\frac{\pi (t - t_0)}{P}\right)$$

where $c_0$ is the solar minimum floor, $A$ is cycle amplitude, $P$ is period (years), and $t_0$ is phase shift.

In [ ]:
csv_path = '../Datasets/sunspots.csv' if os.path.exists('../Datasets/sunspots.csv') else 'Datasets/sunspots.csv'
raw_df = pd.read_csv(csv_path)
date_col = [col for col in raw_df.columns if 'date' in col.lower()][0]
val_col = [col for col in raw_df.columns if col != date_col][-1]

dates = pd.to_datetime(raw_df[date_col])
years = (dates.dt.year + (dates.dt.month - 0.5) / 12.0).to_numpy()
y_obs = pd.to_numeric(raw_df[val_col], errors='coerce').to_numpy()

def harmonic_cycle(t, c0, amp, period, t0):
    return c0 + amp * (np.sin(np.pi * (t - t0) / period) ** 2)

best_sse, init_p = np.inf, None
for p_try in np.linspace(10.0, 12.4, 25):
    for t0_try in np.linspace(0.0, p_try, 20):
        sse = np.sum((y_obs - harmonic_cycle(years, 5.0, y_obs.mean() * 1.7, p_try, t0_try)) ** 2)
        if sse < best_sse:
            best_sse = sse
            init_p = [5.0, y_obs.mean() * 1.7, p_try, t0_try]

popt, _ = optimize.curve_fit(
    harmonic_cycle,
    years,
    y_obs,
    p0=init_p,
    bounds=([0.0, 1.0, 8.0, -20.0], [100.0, 500.0, 15.0, 20.0]),
    maxfev=25000,
)
c0_hat, amp_hat, period_hat, t0_hat = popt
y_fitted = harmonic_cycle(years, *popt)
residuals = y_obs - y_fitted
rmse = np.sqrt(np.mean(residuals ** 2))
r2 = 1.0 - np.sum(residuals ** 2) / np.sum((y_obs - y_obs.mean()) ** 2)

print(f'Baseline c0    : {c0_hat:.2f}')
print(f'Amplitude A    : {amp_hat:.2f}')
print(f'Cycle Period P : {period_hat:.3f} years')
print(f'Phase Shift t0 : {t0_hat:.3f}')
print(f'RMSE           : {rmse:.2f}')
print(f'R-squared      : {r2:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={'width_ratios': [2.2, 1]})
axes[0].plot(years, y_obs, color='#bdbdbd', linewidth=0.75, label='Observed')
axes[0].plot(years, y_fitted, color='#d95f02', linewidth=1.25, label='Deterministic Model')
axes[0].set_title('Deterministic Harmonic Model of the Sunspot Cycle', fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Sunspot Number')
axes[0].legend()

axes[1].hist(residuals, bins=40, color='#7570b3', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Residuals (Observed - Fitted)', fontweight='bold')
axes[1].set_xlabel('Residual')
plt.tight_layout()
plt.show()

print(f'Mean Residual: {residuals.mean():.2f}')

### 5-Year Forecast & Deterministic Reproducibility Check

In [ ]:
future_t = np.arange(years[-1] + 1/12, years[-1] + 5 + 1/12, 1/12)
pred_1 = harmonic_cycle(future_t, *popt)
pred_2 = harmonic_cycle(future_t, *popt)
is_deterministic = bool(np.array_equal(pred_1, pred_2))

forecast_df = pd.DataFrame({
    'Year': np.round(future_t[::12], 2),
    'Predicted Count': np.round(pred_1[::12], 2),
})
print('5-Year Annualized Forecast:')
print(forecast_df.to_string(index=False))
print(f'Identical Output Verification: {is_deterministic}')